# 💬 Notebook 2 — Chatbot Interattivo con ipywidgets

In questo notebook costruiamo una **mini-interfaccia chat** direttamente nel browser,
usando i widget di Jupyter. Gli studenti possono scrivere messaggi e ricevere risposte
senza toccare il codice Python.

---

In [ ]:
import openvino_genai as ov_genai
import ipywidgets as widgets
from IPython.display import display, HTML
import time

# Carica il modello (riuso dalla sessione precedente se già in memoria)
MODEL_DIR = "/workspace/models/tinyllama-chat-ov"

try:
    pipe  # già caricato?
    print("✅ Modello già in memoria")
except NameError:
    print("Caricamento modello...")
    pipe = ov_genai.LLMPipeline(MODEL_DIR, "CPU")
    print("✅ Pronto!")

## 🏗️ Costruiamo il Chatbot

La cella qui sotto crea l'interfaccia grafica. Eseguila e usa la chat qui sotto!

In [ ]:
# ── Storico della conversazione (multi-turn) ─────────────
storico_messaggi = []

SYSTEM_PROMPT = """Sei un assistente AI didattico, utile e conciso.
Rispondi sempre in italiano. Sii chiaro e usa esempi semplici."""

def costruisci_prompt(storico, nuovo_messaggio):
    """Costruisce il prompt completo con tutto lo storico della chat."""
    prompt = f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
    for turno in storico:
        prompt += f"<|user|>\n{turno['user']}</s>\n"
        prompt += f"<|assistant|>\n{turno['assistant']}</s>\n"
    prompt += f"<|user|>\n{nuovo_messaggio}</s>\n<|assistant|>\n"
    return prompt

# ── Widget UI ─────────────────────────────────────────────
area_chat = widgets.Output(
    layout=widgets.Layout(height='380px', overflow_y='auto',
                          border='1px solid #ddd', padding='12px',
                          border_radius='8px', background='#fafafa')
)
campo_testo = widgets.Text(
    placeholder='Scrivi il tuo messaggio...',
    layout=widgets.Layout(width='75%', height='36px')
)
bottone_invia = widgets.Button(
    description='Invia ↵',
    button_style='primary',
    layout=widgets.Layout(width='12%', height='36px')
)
bottone_reset = widgets.Button(
    description='Reset 🗑️',
    button_style='warning',
    layout=widgets.Layout(width='11%', height='36px')
)
stato = widgets.Label(value='🟢 Pronto')

def mostra_messaggio(ruolo, testo):
    colore = '#0066cc' if ruolo == 'Tu' else '#1a7a1a'
    icona  = '👤' if ruolo == 'Tu' else '🤖'
    with area_chat:
        display(HTML(
            f'<div style="margin:8px 0">'
            f'<b style="color:{colore}">{icona} {ruolo}:</b> '
            f'<span style="color:#222">{testo}</span>'
            f'</div>'
        ))

def on_invia(b):
    testo = campo_testo.value.strip()
    if not testo:
        return
    
    campo_testo.value = ''
    campo_testo.disabled = True
    bottone_invia.disabled = True
    stato.value = '⏳ Generazione in corso...'
    
    mostra_messaggio('Tu', testo)
    
    # Genera risposta
    prompt = costruisci_prompt(storico_messaggi, testo)
    cfg = ov_genai.GenerationConfig()
    cfg.max_new_tokens = 256
    cfg.temperature    = 0.7
    cfg.top_p          = 0.9
    cfg.do_sample      = True
    
    tokens = []
    def streamer(token):
        tokens.append(token)
        return False
    
    t0 = time.time()
    pipe.generate(prompt, cfg, streamer)
    elapsed = time.time() - t0
    
    risposta = ''.join(tokens).strip()
    storico_messaggi.append({'user': testo, 'assistant': risposta})
    
    mostra_messaggio('AI', risposta)
    
    campo_testo.disabled = False
    bottone_invia.disabled = False
    stato.value = f'🟢 Pronto  ({len(tokens)} token in {elapsed:.1f}s)'

def on_reset(b):
    storico_messaggi.clear()
    area_chat.clear_output()
    stato.value = '🟢 Pronto — Chat resettata'

bottone_invia.on_click(on_invia)
bottone_reset.on_click(on_reset)
campo_testo.on_submit(on_invia)  # Invio da tastiera

# ── Layout finale ─────────────────────────────────────────
display(HTML('<h3 style="margin-bottom:8px">💬 Chatbot AI Locale</h3>'))
display(area_chat)
display(widgets.HBox([campo_testo, bottone_invia, bottone_reset]))
display(stato)

## 🔍 Ispeziona lo Storico della Conversazione

Esegui questa cella per vedere come viene rappresentata la conversazione internamente:

In [ ]:
import json
print(f"Messaggi nello storico: {len(storico_messaggi)}")
for i, turno in enumerate(storico_messaggi):
    print(f"\n--- Turno {i+1} ---")
    print(f"👤 User:  {turno['user'][:80]}")
    print(f"🤖 AI:    {turno['assistant'][:80]}...")